In [1]:
# Sel ini menghasilkan TIGA dataset cabang (kota) terpisah untuk Tugas Mandiri Pertemuan 3
import numpy as np
import pandas as pd

kategori_list = ["Elektronik", "Fashion", "Makanan & Minuman", "Kesehatan & Kecantikan", "Rumah Tangga"]
metode_bayar_list = ["Transfer Bank", "E-Wallet", "COD", "Kartu Kredit"]
tanggal_range = pd.date_range("2026-08-01", "2026-08-31", freq="D")

cabang_kota = {"Magelang": 101, "Yogyakarta": 202, "Semarang": 303}

for kota, seed in cabang_kota.items():
    np.random.seed(seed)  # seed berbeda tiap kota agar datanya bervariasi, namun tetap konsisten/reproducible
    n = 200
    data_cabang = {
        "order_id": [f"{kota[:3].upper()}-{2000 + i}" for i in range(n)],
        "tanggal": np.random.choice(tanggal_range, size=n),
        "kategori": np.random.choice(kategori_list, size=n, p=[0.25, 0.25, 0.20, 0.15, 0.15]),
        "unit_terjual": np.random.randint(1, 8, size=n),
        "harga_satuan": np.random.choice([25000, 50000, 75000, 100000, 150000, 250000], size=n),
        "metode_pembayaran": np.random.choice(metode_bayar_list, size=n),
    }
    df_cabang = pd.DataFrame(data_cabang)
    df_cabang["kota"] = kota
    nama_file = f"transaksi_{kota.lower()}.csv"
    df_cabang.to_csv(nama_file, index=False)
    print(f"Berkas '{nama_file}' berhasil dibuat: {df_cabang.shape[0]} baris")

print("\nKetiga berkas CSV cabang siap digunakan untuk Tugas Mandiri.")

Berkas 'transaksi_magelang.csv' berhasil dibuat: 200 baris
Berkas 'transaksi_yogyakarta.csv' berhasil dibuat: 200 baris
Berkas 'transaksi_semarang.csv' berhasil dibuat: 200 baris

Ketiga berkas CSV cabang siap digunakan untuk Tugas Mandiri.


In [7]:
# Buat direktori baru
!hdfs dfs -mkdir -p /user/mazedev/ecommerce/raw
!hdfs dfs -mkdir -p /user/mazedev/ecommerce/processed

#Cek direktori untuk verifikasi
!hdfs dfs -ls /user/mazedev

Found 4 items
drwxr-xr-x   - mazedev supergroup          0 2026-09-04 20:28 /user/mazedev/ecommerce
drwxr-xr-x   - mazedev supergroup          0 2026-09-04 18:53 /user/mazedev/latihan3
drwxr-xr-x   - mazedev supergroup          0 2026-09-04 18:50 /user/mazedev/praktikum
drwxr-xr-x   - mazedev supergroup          0 2026-09-04 20:24 /user/mazedev/user


In [12]:
!hdfs dfs -rm -r /user/mazedev/user
#Cek direktori untuk verifikasi
!hdfs dfs -ls /user/mazedev

Found 4 items
drwxr-xr-x   - mazedev supergroup          0 2026-09-04 20:28 /user/mazedev/ecommerce
drwxr-xr-x   - mazedev supergroup          0 2026-09-04 18:53 /user/mazedev/latihan3
drwxr-xr-x   - mazedev supergroup          0 2026-09-04 18:50 /user/mazedev/praktikum
drwxr-xr-x   - mazedev supergroup          0 2026-09-04 20:24 /user/mazedev/user


In [1]:
#Unggah Data mentah ke HDFS
!hdfs dfs -put transaksi_magelang.csv transaksi_yogyakarta.csv transaksi_semarang.csv /user/mazedev/ecommerce/raw

#Tampilkan direktori untuk verifikasi
!hdfs dfs -ls /user/mazedev/ecommerce/raw

Found 3 items
-rw-r--r--   1 mazedev supergroup      12329 2026-09-05 08:15 /user/mazedev/ecommerce/raw/transaksi_magelang.csv
-rw-r--r--   1 mazedev supergroup      12154 2026-09-05 08:15 /user/mazedev/ecommerce/raw/transaksi_semarang.csv
-rw-r--r--   1 mazedev supergroup      12684 2026-09-05 08:15 /user/mazedev/ecommerce/raw/transaksi_yogyakarta.csv


In [6]:
#Membaca Kembali dan Menggabugnkan Data dari HDFS
# Memerlukan library io, subprocess, dan pandas
import io
import subprocess as subp
import pandas as pd

# Cek versi library
print("pandas      :", pd.__version__)

# Menentukan path file di HDFS
hdfs_path_magelang = "/user/mazedev/ecommerce/raw/transaksi_magelang.csv"
hdfs_path_yogyakarta = "/user/mazedev/ecommerce/raw/transaksi_yogyakarta.csv"
hdfs_path_semarang = "/user/mazedev/ecommerce/raw/transaksi_semarang.csv"

pandas      : 2.2.3


In [7]:
# Fungsi helper untuk membaca berkas dari HDFS langsung ke DataFrame
def read_hdfs_csv(hdfs_path):
    # Menjalankan perintah 'hdfs dfs -cat [path]' via subprocess
    cmd = ["hdfs", "dfs", "-cat", hdfs_path]
    output = subp.check_output(cmd)

    # Mengubah bytes output menjadi StringISO agar dapat dibaca pd.read_csv
    return pd.read_csv(io.StringIO(output.decode("utf-8")))

In [30]:
# Membaca masing-masing dataset dari HDFS
df_magelang = read_hdfs_csv(hdfs_path_magelang)
df_yogyakarta = read_hdfs_csv(hdfs_path_yogyakarta)
df_semarang = read_hdfs_csv(hdfs_path_semarang)

In [31]:
# Verifikasi dataset magelang
df_magelang

,order_id,tanggal,kategori,unit_terjual,harga_satuan,metode_pembayaran,kota
0,MAG-2000,2026-08-12,Fashion,1,50000,Transfer Bank,Magelang
1,MAG-2001,2026-08-18,Elektronik,7,25000,COD,Magelang
2,MAG-2002,2026-08-07,Elektronik,7,25000,E-Wallet,Magelang
3,MAG-2003,2026-08-24,Rumah Tangga,6,25000,COD,Magelang
4,MAG-2004,2026-08-30,Fashion,7,100000,Transfer Bank,Magelang
...,...,...,...,...,...,...,...
195,MAG-2195,2026-08-09,Makanan & Minuman,1,250000,COD,Magelang
196,MAG-2196,2026-08-28,Fashion,6,250000,COD,Magelang
197,MAG-2197,2026-08-14,Elektronik,1,250000,COD,Magelang
198,MAG-2198,2026-08-02,Makanan & Minuman,6,100000,Transfer Bank,Magelang


In [32]:
# Verifikasi dataset yogyakarta
df_yogyakarta

,order_id,tanggal,kategori,unit_terjual,harga_satuan,metode_pembayaran,kota
0,YOG-2000,2026-08-20,Fashion,5,250000,E-Wallet,Yogyakarta
1,YOG-2001,2026-08-24,Fashion,4,150000,Kartu Kredit,Yogyakarta
2,YOG-2002,2026-08-29,Rumah Tangga,4,100000,E-Wallet,Yogyakarta
3,YOG-2003,2026-08-13,Fashion,7,250000,E-Wallet,Yogyakarta
4,YOG-2004,2026-08-09,Elektronik,7,75000,COD,Yogyakarta
...,...,...,...,...,...,...,...
195,YOG-2195,2026-08-25,Rumah Tangga,4,250000,Kartu Kredit,Yogyakarta
196,YOG-2196,2026-08-18,Fashion,1,150000,COD,Yogyakarta
197,YOG-2197,2026-08-27,Makanan & Minuman,4,25000,Kartu Kredit,Yogyakarta
198,YOG-2198,2026-08-07,Rumah Tangga,1,25000,COD,Yogyakarta


In [33]:
# Verifikasi dataset semarang
df_semarang

,order_id,tanggal,kategori,unit_terjual,harga_satuan,metode_pembayaran,kota
0,SEM-2000,2026-08-28,Elektronik,3,100000,E-Wallet,Semarang
1,SEM-2001,2026-08-29,Kesehatan & Kecantikan,4,25000,Transfer Bank,Semarang
2,SEM-2002,2026-08-19,Rumah Tangga,1,25000,COD,Semarang
3,SEM-2003,2026-08-02,Makanan & Minuman,2,75000,E-Wallet,Semarang
4,SEM-2004,2026-08-05,Elektronik,6,25000,Transfer Bank,Semarang
...,...,...,...,...,...,...,...
195,SEM-2195,2026-08-31,Rumah Tangga,6,75000,Transfer Bank,Semarang
196,SEM-2196,2026-08-15,Kesehatan & Kecantikan,5,75000,Transfer Bank,Semarang
197,SEM-2197,2026-08-01,Makanan & Minuman,4,75000,E-Wallet,Semarang
198,SEM-2198,2026-08-05,Fashion,7,25000,Transfer Bank,Semarang


In [34]:
# Menggabungkan ketiga DataFrame menjadi satu
df_gabungan = pd.concat(
    [df_magelang, df_yogyakarta, df_semarang], ignore_index=True
)

#Tampilkan DataFrame gabungan
df_gabungan

,order_id,tanggal,kategori,unit_terjual,harga_satuan,metode_pembayaran,kota
0,MAG-2000,2026-08-12,Fashion,1,50000,Transfer Bank,Magelang
1,MAG-2001,2026-08-18,Elektronik,7,25000,COD,Magelang
2,MAG-2002,2026-08-07,Elektronik,7,25000,E-Wallet,Magelang
3,MAG-2003,2026-08-24,Rumah Tangga,6,25000,COD,Magelang
4,MAG-2004,2026-08-30,Fashion,7,100000,Transfer Bank,Magelang
...,...,...,...,...,...,...,...
595,SEM-2195,2026-08-31,Rumah Tangga,6,75000,Transfer Bank,Semarang
596,SEM-2196,2026-08-15,Kesehatan & Kecantikan,5,75000,Transfer Bank,Semarang
597,SEM-2197,2026-08-01,Makanan & Minuman,4,75000,E-Wallet,Semarang
598,SEM-2198,2026-08-05,Fashion,7,25000,Transfer Bank,Semarang


In [37]:
# Pembuktian hasil gabungan berisi transaksi 3 kota (menghitung value kolom kota)
df_gabungan["kota"].value_counts()

kota
Magelang      200
Yogyakarta    200
Semarang      200
Name: count, dtype: int64

In [38]:
# Menambahkan kolom total_pendapatan (unit_terjual x harga_satuan)

In [40]:
df_gabungan["total_pendapatan"] = df_gabungan["unit_terjual"] * df_gabungan["harga_satuan"]
df_gabungan

,order_id,tanggal,kategori,unit_terjual,harga_satuan,metode_pembayaran,kota,total_pendapatan
0,MAG-2000,2026-08-12,Fashion,1,50000,Transfer Bank,Magelang,50000
1,MAG-2001,2026-08-18,Elektronik,7,25000,COD,Magelang,175000
2,MAG-2002,2026-08-07,Elektronik,7,25000,E-Wallet,Magelang,175000
3,MAG-2003,2026-08-24,Rumah Tangga,6,25000,COD,Magelang,150000
4,MAG-2004,2026-08-30,Fashion,7,100000,Transfer Bank,Magelang,700000
...,...,...,...,...,...,...,...,...
595,SEM-2195,2026-08-31,Rumah Tangga,6,75000,Transfer Bank,Semarang,450000
596,SEM-2196,2026-08-15,Kesehatan & Kecantikan,5,75000,Transfer Bank,Semarang,375000
597,SEM-2197,2026-08-01,Makanan & Minuman,4,75000,E-Wallet,Semarang,300000
598,SEM-2198,2026-08-05,Fashion,7,25000,Transfer Bank,Semarang,175000


In [43]:
# Membuat tabel ringkasan (total pendapatan per kota dan per kategori)
ringkasan = df_gabungan.groupby(["kota", "kategori"])["total_pendapatan"].sum().reset_index()
ringkasan

,kota,kategori,total_pendapatan
0,Magelang,Elektronik,18775000
1,Magelang,Fashion,27750000
2,Magelang,Kesehatan & Kecantikan,17375000
3,Magelang,Makanan & Minuman,17525000
4,Magelang,Rumah Tangga,12200000
5,Semarang,Elektronik,21425000
6,Semarang,Fashion,26425000
7,Semarang,Kesehatan & Kecantikan,13525000
8,Semarang,Makanan & Minuman,19750000
9,Semarang,Rumah Tangga,10975000


In [46]:
# Simpan DataFrame ke file lokal
df_gabungan.to_csv("data_gabungan.csv", index=False)
ringkasan.to_csv("ringkasan_kota_kategori.csv", index=False)

# Verifikasi apakah sudah berhasil
!ls -la data_gabungan.csv ringkasan_kota_kategori.csv

-rw-rw-r-- 1 mazedev mazedev 41257 Sep  5 09:16 data_gabungan.csv
-rw-rw-r-- 1 mazedev mazedev   530 Sep  5 09:16 ringkasan_kota_kategori.csv


In [47]:
# Unggah ke hdfs
!hdfs dfs -put data_gabungan.csv /user/mazedev/ecommerce/processed
!hdfs dfs -put ringkasan_kota_kategori.csv /user/mazedev/ecommerce/processed

# Verifikasi direktori
!hdfs dfs -ls /user/mazedev/ecommerce/processed

Found 2 items
-rw-r--r--   1 mazedev supergroup      41257 2026-09-05 09:20 /user/mazedev/ecommerce/processed/data_gabungan.csv
-rw-r--r--   1 mazedev supergroup        530 2026-09-05 09:20 /user/mazedev/ecommerce/processed/ringkasan_kota_kategori.csv
